# Validating State Estimation Using Extended Kalman Filter (EKF)

### Setup Environement

In [1]:
!pip uninstall -y energy-plus-utility

Found existing installation: energy-plus-utility 0.2.2+5
Uninstalling energy-plus-utility-0.2.2+5:
  Successfully uninstalled energy-plus-utility-0.2.2+5


In [2]:
!pip install -q "energy-plus-utility @ git+https://github.com/janithcyapa/energy-plus-utility.git@main"
import importlib.metadata
ver = importlib.metadata.version("energy-plus-utility")
print(f"\n✅ Installed 'energy-plus-utility' version: {ver}")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

✅ Installed 'energy-plus-utility' version: 0.2.2+5


In [3]:
from eplus import prepare_colab_eplus
prepare_colab_eplus(silent=False)

In [4]:
!pip install control

## Setup Model

In [5]:
# @title Import Packages
import types
import datetime
import traceback
import requests
import io
import gc
import os
from pathlib import Path
import re

from eplus.core import EPlusUtil
import pandas as pd
import numpy as np
import control as ct
import scipy.sparse as sparse
import osqp
import scipy.linalg as la


import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

In [6]:
# @title Setup Model
OUT_DIR = "/simulation/eplus_out"
# url_idf ="file:///tmp/modified_5ZoneAirCooled.idf"
url_idf="https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/5ZoneAirCooled.idf"
url_epw = "https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/Weather%20Files/LKA_Colombo-Katunayake.434500_SWERA.epw"

# Initialize Utility

sim = EPlusUtil(verbose=0, out_dir=OUT_DIR)
sim.reset_state()
sim.delete_out_dir()
sim.clear_eplus_outputs(patterns="eplusout.*")
sim.set_model_from_url(url_idf, url_epw)

In [7]:
# @title Update Model
with open(sim.idf, 'r', encoding='utf-8') as f:
    idf_text = f.read()

# 2. Use the utility to safely remove ALL old Location and Design Days
idf_text = sim._remove_object_blocks(idf_text, "Site:Location")
idf_text = sim._remove_object_blocks(idf_text, "SizingPeriod:DesignDay")

# 3. Surgically remove ONLY the 3 schedules we want to override using Regex
for sched in ["FanAvailSched", "CoolingCoilAvailSched", "ReheatCoilAvailSched"]:
    pattern = re.compile(rf"(?i)^\s*Schedule:Compact[,\s]+{sched}[,\s].*?;[^\n]*\n?", re.MULTILINE | re.DOTALL)
    idf_text = pattern.sub("\n", idf_text)

# 4. Fetch the Colombo .ddy file
url_ddy = "https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/Weather%20Files/LKA_Colombo-Katunayake.434500_SWERA.ddy"
resp = requests.get(url_ddy)
ddy_text = resp.text

# 5. Define the 24/7 Schedules
new_schedules = """
Schedule:Compact, FanAvailSched, Fraction, Through: 12/31, For: AllDays, Until: 24:00, 1.0;
Schedule:Compact, CoolingCoilAvailSched, Fraction, Through: 12/31, For: AllDays, Until: 24:00, 1.0;
Schedule:Compact, ReheatCoilAvailSched, Fraction, Through: 12/31, For: AllDays, Until: 24:00, 1.0;
"""

# 6. Append the DDY text and the new schedules to the IDF
idf_text = sim._append_block(idf_text, ddy_text)
idf_text = sim._append_block(idf_text, new_schedules)

# 7. Overwrite the file in place
with open(sim.idf, 'w', encoding='utf-8') as f:
    f.write(idf_text)

print("IDF safely patched with Colombo DDY and 24/7 schedules!")



IDF safely patched with Colombo DDY and 24/7 schedules!


In [8]:
# @title Run Dry Day

sim.ensure_output_sqlite()
sim.prepare_run_with_co2(
    outdoor_co2_ppm=420.0,
    wipe_outputs=True,
    activate=True,
    reset=True
)

print("Executing Dry Run...")
sim.run_dry_run(include_ems_edd=False, reset=True, design_day=True)
print("Dry Run Complete!")



# catalog = sim.api_catalog_df()
# mo.ui.table(catalog)
# mo.ui.table(catalog['VARIABLES'])
# sim.list_available_variables()

Executing Dry Run...
Dry Run Complete!


## Setup Simulator

In [9]:
# @title Setup Data Logger
# Request the variables to construct State Vector (x_i) and Disturbances (d_i)
specs = [
    # Zone States (x_i)
    {"name": "Zone Mean Air Temperature", "key": "*"},       # T_in,i
    {"name": "Zone Mean Radiant Temperature", "key": "*"},   # T_m,i (Thermal Mass proxy)
    {"name": "Zone Mean Air Humidity Ratio", "key": "*"},    # W_in,i
    {"name": "Zone Air Relative Humidity", "key": "*"},      # W_in,i (%)
    {"name": "Zone Air CO2 Concentration", "key": "*"},      # ppm

    # Time-Varying Parameters (p_i)
    {"name": "Zone People Occupant Count", "key": "*"},      # No. of People
    {"name": "Zone Electric Equipment Total Heating Rate", "key": "*"}, # Watts

    # External Environment Conditions (x_out)
    {"name": "Site Outdoor Air Drybulb Temperature", "key": "*"}, # T_out
    {"name": "Site Outdoor Air Humidity Ratio", "key": "*"},      # W_out
    {"name": "Site Outdoor Air Relative Humidity", "key": "*"},   # W_in,i (%)
    {"name": "Schedule Value", "key": "CO2-Outdoor-Actuated"},     # ppm

    # Control Inputs - VAV Box Volumetric Flow Rate (m3/s)
    {"name": "System Node Current Density Volume Flow Rate", "key": "*"},

    # AHU Supply Parameters (S)
    {"name": "System Node Temperature", "key": "*"},       # T_s
    {"name": "System Node Humidity Ratio", "key": "*"},    # W_s
    {"name": "System Node CO2 Concentration", "key": "*"}, # C_s
]
sim.ensure_output_variables(specs, activate=True)

sim.collected_data = []
sim.current_state = {}

def state_logger(self, state):
    """Extracts sensor data, updates the current snapshot, and logs history."""
    if not self.exchange.api_data_fully_ready(state):
        return

    # 1. Get Simulation Time Details
    day = self.exchange.day_of_year(state)
    time_now = self.exchange.current_time(state)
    total_minutes = int(time_now * 60)
    hours, mins = divmod(total_minutes, 60)

    row = {
        "timestamp": f"Day {day:03d} {hours:02d}:{mins:02d}",
        "day": day,
        "hour": hours,
        "minute": mins,
        "time_decimal": time_now
    }

    # 2. Extract Outdoor and Supply Data
    t_out_h = self.exchange.get_variable_handle(state, "Site Outdoor Air Drybulb Temperature", "Environment")
    w_out_h = self.exchange.get_variable_handle(state, "Site Outdoor Air Humidity Ratio", "Environment")
    rh_out_h = self.exchange.get_variable_handle(state, "Site Outdoor Air Relative Humidity", "Environment")
    co2_out_h = self.exchange.get_variable_handle(state, "Schedule Value", "CO2-Outdoor-Actuated")

    row["T_out"] = self.exchange.get_variable_value(state, t_out_h)
    row["W_out"] = self.exchange.get_variable_value(state, w_out_h)
    row["RH_out_%"] = self.exchange.get_variable_value(state, rh_out_h)
    row["CO2_out"] = self.exchange.get_variable_value(state, co2_out_h)


    t_s_h = self.exchange.get_variable_handle(state, "System Node Temperature", "VAV Sys 1 Outlet Node")
    w_s_h = self.exchange.get_variable_handle(state, "System Node Humidity Ratio", "VAV Sys 1 Outlet Node")
    c_s_h = self.exchange.get_variable_handle(state, "System Node CO2 Concentration", "VAV Sys 1 Outlet Node")

    row["T_s"] = self.exchange.get_variable_value(state, t_s_h)
    row["W_s"] = self.exchange.get_variable_value(state, w_s_h)
    row["C_s"] = self.exchange.get_variable_value(state, c_s_h)

    # 3. Extract Internal States for All Zones
    zones = ["SPACE1-1", "SPACE2-1", "SPACE3-1", "SPACE4-1", "SPACE5-1"]
    for zone in zones:
        t_in_h = self.exchange.get_variable_handle(state, "Zone Mean Air Temperature", zone)
        t_m_h = self.exchange.get_variable_handle(state, "Zone Mean Radiant Temperature", zone)
        w_in_h = self.exchange.get_variable_handle(state, "Zone Mean Air Humidity Ratio", zone)
        rh_in_h = self.exchange.get_variable_handle(state, "Zone Air Relative Humidity", zone)
        co2_in_h = self.exchange.get_variable_handle(state, "Zone Air CO2 Concentration", zone)
        occ_in_h = self.exchange.get_variable_handle(state, "Zone People Occupant Count", zone)
        q_eq_h = self.exchange.get_variable_handle(state, "Zone Electric Equipment Total Heating Rate", zone)
        v_dot_h = self.exchange.get_variable_handle(state, "System Node Current Density Volume Flow Rate", f"{zone} In Node")

        row[f"{zone}_T_in"] = self.exchange.get_variable_value(state, t_in_h)
        row[f"{zone}_T_m"] = self.exchange.get_variable_value(state, t_m_h)
        row[f"{zone}_W_in"] = self.exchange.get_variable_value(state, w_in_h)
        row[f"{zone}_RH_%"] = self.exchange.get_variable_value(state, rh_in_h)
        row[f"{zone}_CO2_in"] = self.exchange.get_variable_value(state, co2_in_h)
        row[f"{zone}_Occ"] = self.exchange.get_variable_value(state, occ_in_h)
        row[f"{zone}_Q_equip"] = self.exchange.get_variable_value(state, q_eq_h)
        row[f"{zone}_V_dot"] = self.exchange.get_variable_value(state, v_dot_h)

    # 4. Update the current snapshot AND append to historical log
    self.current_state = row
    self.collected_data.append(row)

sim.state_logger = types.MethodType(state_logger, sim)
sim.register_handlers(
    "begin", [
        {"method_name": "state_logger"},
        # {"method_name": "occupancy_handler", "kwargs": {"lam": 3.0, "min": 0, "max": 5, "seed": 4 } },
        {"method_name": "co2_set_outdoor_ppm", "kwargs": { "value_ppm": 420.0, "log_every_minutes": 60 } }
    ]
)

print(f"Handlers on 'begin' hook: {sim.list_handlers("begin")}")

Handlers on 'begin' hook: ['state_logger', 'co2_set_outdoor_ppm']


In [10]:
# @title occupancy_csv

def preload_occupancy_csv(sim_obj, url):
    """
    Downloads the CSV, anchors time to midnight, and forces a perfect 24-hour loop.
    """
    print(f"Downloading CSV from: {url}...")
    try:
        resp = requests.get(url)
        resp.raise_for_status()

        df = pd.read_csv(io.StringIO(resp.text))
        if 'timestamp' not in df.columns:
            raise ValueError("CSV must contain a 'timestamp' column.")

        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df = df.sort_values('timestamp')

        # 1. Anchor to MIDNIGHT of the first day to prevent timestep offset
        midnight_start = df['timestamp'].iloc[0].normalize()
        df['rel_seconds'] = (df['timestamp'] - midnight_start).dt.total_seconds()

        # 2. Force exactly 24 hours for daily looping to prevent modulo drift
        sim_obj._occ_duration_sec = 86400.0

        # 3. Clean up dataframe
        df = df.set_index('rel_seconds')
        sim_obj._preloaded_occ_df = df.drop(columns=['timestamp'])

        zones = list(sim_obj._preloaded_occ_df.columns)
        print(f"Success! Preloaded {len(df)} rows. Loop locked to 24.00 hours.")
        print(f"Detected Source Columns: {zones}")

    except Exception as e:
        print(f"Failed to preload CSV: {e}")
# Load CSV
csv_url = "https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/Weather%20Files/Occupancy_Dataset.csv"
preload_occupancy_csv(sim, csv_url)

def people_injector(self, state):
    """
    Lightning-fast runtime handler. Maps actuators on the first tick.
    Reads a single baseline zone from the CSV and uses multipliers,
    ceilings, and clipping to populate other zones synthetically.
    """
    if not self.exchange.api_data_fully_ready(state) or self.exchange.warmup_flag(state):
        return

    # --- 1. One-Time Setup: Map Actuators & Define Extrapolation Rules ---
    if not hasattr(self, '_fast_injector_ready'):
        # Ensure the data was preloaded via Step 1
        if not hasattr(self, '_preloaded_occ_df'):
            print("[Injector] ERROR: Data not preloaded. Run preload_occupancy_csv() first.")
            self._fast_injector_ready = False
            return

        # =========================================================
        # CONFIGURATION DICTIONARY: TWEAK YOUR MULTIPLIERS HERE
        # source: The CSV column to read the baseline value from
        # mult: The multiplier applied to the source value
        # min/max: The clipping bounds to enforce physical limits
        # =========================================================
        self._zone_occ_rules = {
            "SPACE1-1": {"source": "SPACE1-1", "mult": 1.0,  "min": 0, "max": 5}, # Baseline
            "SPACE2-1": {"source": "SPACE1-1", "mult": 1.5,  "min": 0, "max": 4},
            "SPACE3-1": {"source": "SPACE1-1", "mult": 0.4,  "min": 0, "max": 1},
            "SPACE4-1": {"source": "SPACE1-1", "mult": 1.2,  "min": 0, "max": 3},
            "SPACE5-1": {"source": "SPACE1-1", "mult": 2.0,  "min": 0, "max": 6},
        }

        self._people_handles = {}
        target_zones = list(self._zone_occ_rules.keys())

        try:
            ep_people_names = self.exchange.get_object_names(state, "People") or []
        except Exception:
            ep_people_names = []

        # Map to EnergyPlus Actuators based on the target zones, NOT just CSV columns
        mapped_count = 0
        for z in target_zones:
            matched_people = [p for p in ep_people_names if z.replace(" ", "").lower() in p.replace(" ", "").lower()]
            handles = []
            for p in matched_people:
                h = self.exchange.get_actuator_handle(state, "People", "Number of People", p)
                if h != -1:
                    handles.append(h)
                    mapped_count += 1
            if handles:
                self._people_handles[z] = handles

        print(f"\n[Injector] Mapped {mapped_count} actuators across {len(self._people_handles)} zones (Extrapolation Active).")

        # Mark exact start time of the simulation
        day = self.exchange.day_of_year(state)
        time_hr = self.exchange.current_time(state)
        self._sim_start_date = datetime.datetime(2002, 1, 1) + datetime.timedelta(days=day - 1, seconds=(int(time_hr * 3600)))

        self._fast_injector_ready = True

    # --- Runtime Safety Check ---
    if not self._fast_injector_ready or getattr(self, '_occ_duration_sec', 0) == 0:
        return

    # --- 2. Calculate Elapsed Time & Loop ---
    day = self.exchange.day_of_year(state)
    time_hr = self.exchange.current_time(state)
    current_date = datetime.datetime(2002, 1, 1) + datetime.timedelta(days=day - 1, seconds=(int(time_hr * 3600)))

    elapsed_seconds = (current_date - self._sim_start_date).total_seconds()
    loop_sec = elapsed_seconds % self._occ_duration_sec

    # --- 3. Fast Data Lookup (Forward Fill) ---
    df = self._preloaded_occ_df
    valid_indices = df.index[df.index <= loop_sec]
    target_idx = df.index[0] if len(valid_indices) == 0 else valid_indices[-1]
    row = df.loc[target_idx]

    # --- 4. Extrapolate and Inject Values ---
    for z, handles in self._people_handles.items():
        rule = self._zone_occ_rules.get(z)
        if not rule:
            continue

        src_col = rule["source"]
        if src_col in row:
            base_val = float(row[src_col])

            # Apply math: Base * Multiplier -> Round Up -> Clip
            if base_val == 0:
                val = 0.0 # Bypasses math to strictly enforce zero at night
            else:
                calculated = np.ceil(base_val * rule["mult"])
                val = float(np.clip(calculated, rule["min"], rule["max"]))

            # Divide evenly if there are multiple People objects in the same room
            per_actuator = val / len(handles)
            for h in handles:
                self.exchange.set_actuator_value(state, h, per_actuator)

# --- Registration ---
sim.people_injector = types.MethodType(people_injector, sim)

sim.register_handlers("begin", [
    {"method_name": "people_injector"},
])

print(f"Handlers on 'begin' hook: {sim.list_handlers("begin")}")

Success! Preloaded 10129 rows. Loop locked to 24.00 hours.
Detected Source Columns: ['SPACE1-1']
Handlers on 'begin' hook: ['state_logger', 'co2_set_outdoor_ppm', 'people_injector']


In [11]:
# @title zone_model
def zone_model(self, state):
    """
    Forward-simulate one timestep for every thermal zone using the RC model.
    - works for all zones automatically
    - uses consistent CO₂ units (volumetric fraction)
    - logs actual vs. predicted states
    """
    try:
        if not self.exchange.api_data_fully_ready(state):
            return

        # ----- One‑time setup -----
        if not hasattr(self, 'zones'):
            self.zones = {}
            self.zone_list = list(self.get_zone_thermal_parameters().keys())
            print(f"[Zone Model] Active zones: {self.zone_list}")

        day   = self.exchange.day_of_year(state)
        time  = self.exchange.current_time(state)
        base_date = datetime.datetime(2026, 1, 1) + datetime.timedelta(
                        days=day-1, seconds=int(time*3600))

        # ----- Loop over all zones -----
        for zone_id in self.zone_list:

            # ========== Initialise zone ==========
            if zone_id not in self.zones:
                raw_params = self.get_zone_thermal_parameters()[zone_id]

                # Variable handles
                handles = {
                    "T_in": self.exchange.get_variable_handle(state, "Zone Mean Air Temperature", zone_id),
                    "T_m":  self.exchange.get_variable_handle(state, "Zone Mean Radiant Temperature", zone_id),
                    "W_in": self.exchange.get_variable_handle(state, "Zone Mean Air Humidity Ratio", zone_id),
                    "CO2_in": self.exchange.get_variable_handle(state, "Zone Air CO2 Concentration", zone_id),
                    "N_occ": self.exchange.get_variable_handle(state, "Zone People Occupant Count", zone_id),
                    "T_out": self.exchange.get_variable_handle(state, "Site Outdoor Air Drybulb Temperature", "Environment"),
                    "Q_equip": self.exchange.get_variable_handle(state, "Zone Electric Equipment Total Heating Rate", zone_id),
                    "V_dot": self.exchange.get_variable_handle(state, "System Node Current Density Volume Flow Rate", f"{zone_id} In Node"),
                    "T_s": self.exchange.get_variable_handle(state, "System Node Temperature", "VAV Sys 1 Outlet Node"),
                    "W_s": self.exchange.get_variable_handle(state, "System Node Humidity Ratio", "VAV Sys 1 Outlet Node"),
                    "C_s": self.exchange.get_variable_handle(state, "System Node CO2 Concentration", "VAV Sys 1 Outlet Node"),
                }

                # ---- Thermal resistance to outdoors & adjacent zones ----
                inv_R_env_ext = 0.0
                adj_zones = []
                for b in raw_params["boundaries"]:
                    target = b["target"]
                    r_abs  = float(b["R_absolute_K_W"])
                    if target == "Ground":
                        R_env_gnd = r_abs
                    elif target == "Environment" or b["boundary_condition"] == "outdoors":
                        inv_R_env_ext += 1.0 / r_abs
                    else:
                        adj_zones.append({
                            "zone": target,
                            "R_env": r_abs,
                            "handle_T_in": self.exchange.get_variable_handle(state, "Zone Mean Air Temperature", target)
                        })
                R_env_ext = 1.0 / inv_R_env_ext if inv_R_env_ext > 0 else float('inf')
                # (R_env_gnd kept but not used in dynamics as ground temp is assumed 22°C)

                # ----- Nonlinear system definition -----
                def _dynamics(t, x, u, params):
                    T_in, T_m, W_in, C_in = x
                    V_dot_s = float(u[0])

                    # Constants
                    rho_air, cp_air = 1.204, 1006.0
                    q_person   = 100.0          # W per person
                    g_w_person = 5e-5           # kg/s per person (humidity generation)
                    g_co2_person = 1e-5         # m³/s per person (volumetric CO₂ at room conditions)

                    R_env_ext = float(params.get('R_env_ext', float('inf')))
                    R_env_gnd = float(params.get('R_env_gnd', float('inf')))
                    R_int     = float(params['R_int'])
                    C_air     = float(params['C_air'])
                    C_mass    = float(params['C_mass'])
                    M_air     = float(params['M_air'])
                    V_room    = float(params['V_room'])

                    T_s = params['T_s']
                    W_s = params['W_s']
                    C_s = params['C_s']          # already in m³/m³ (volumetric fraction)
                    N_occ   = params['N_occ']
                    Q_equip = params['Q_equip']
                    T_out   = params['T_out']

                    d_T, d_W, d_C = params['d_T'], params['d_W'], params['d_C']

                    # ---- Temperature ----
                    q_env  = (T_out - T_in) / R_env_ext if R_env_ext < float('inf') else 0.0
                    q_gnd  = (22.0 - T_in) / R_env_gnd if R_env_gnd < float('inf') else 0.0
                    q_adj  = sum((float(adj['T_in']) - T_in) / float(adj['R_env']) for adj in params['adj_zones'])
                    q_mass = (T_m - T_in) / R_int
                    q_int  = N_occ * q_person + Q_equip
                    q_s    = rho_air * V_dot_s * cp_air * (T_s - T_in)
                    dT_in_dt = (q_env + q_gnd + q_adj + q_mass + q_int + q_s + d_T) / C_air

                    # ---- Mass temperature ----
                    dT_m_dt = (T_in - T_m) / (C_mass * R_int)

                    # ---- Humidity ratio ----
                    dot_m_s = rho_air * V_dot_s
                    dW_in_dt = (N_occ * g_w_person + dot_m_s * (W_s - W_in) + d_W) / M_air

                    # ---- CO₂ (volumetric fraction) ----
                    dC_in_dt = (N_occ * g_co2_person + V_dot_s * (C_s - C_in) + d_C) / V_room

                    return np.array([dT_in_dt, dT_m_dt, dW_in_dt, dC_in_dt], dtype=float).flatten()

                def _outputs(t, x, u, params):
                    return [x[0], x[1], x[2], x[3]]

                sys_ode = ct.NonlinearIOSystem(
                    _dynamics, _outputs,
                    inputs=['V_dot_s'],
                    outputs=['T_in_obs', 'T_m_obs', 'W_in_obs', 'C_in_obs'],
                    states=['T_in', 'T_m', 'W_in', 'C_in'],
                    name=f'sys_{zone_id}'
                )

                # Store everything
                self.zones[zone_id] = types.SimpleNamespace(
                    V_room   = float(raw_params['V_room']),
                    M_air    = float(raw_params['M_air']),
                    C_air    = float(raw_params['C_air']),
                    C_mass   = float(raw_params['C_mass']),
                    R_int    = float(raw_params['R_int']),
                    R_env_gnd = R_env_gnd if 'R_env_gnd' in locals() else float('inf'),
                    R_env_ext = R_env_ext,
                    adj_zones = adj_zones,
                    handles  = handles,
                    sys_ode  = sys_ode,
                    log      = [],
                    prev_prediction = None   # will store the last predicted next state
                )

                print(f"[{zone_id}] RC model ready. Adjacent: {[a['zone'] for a in adj_zones]}")

            # ========== Time‑step execution ==========
            z = self.zones[zone_id]

            # ---- Read current values ----
            V_dot_s = self.exchange.get_variable_value(state, z.handles["V_dot"])
            T_in_true = self.exchange.get_variable_value(state, z.handles["T_in"])
            T_m_true  = self.exchange.get_variable_value(state, z.handles["T_m"])
            W_in_true = self.exchange.get_variable_value(state, z.handles["W_in"])
            C_in_ppm  = self.exchange.get_variable_value(state, z.handles["CO2_in"])   # ppm

            # Convert CO₂ to volumetric fraction (m³/m³)
            C_in_frac = C_in_ppm * 1e-6

            T_out  = self.exchange.get_variable_value(state, z.handles["T_out"])
            N_occ  = self.exchange.get_variable_value(state, z.handles["N_occ"])
            Q_equip = self.exchange.get_variable_value(state, z.handles["Q_equip"])
            T_s    = self.exchange.get_variable_value(state, z.handles["T_s"])
            W_s    = self.exchange.get_variable_value(state, z.handles["W_s"])
            C_s_ppm = self.exchange.get_variable_value(state, z.handles["C_s"])        # ppm
            C_s_frac = C_s_ppm * 1e-6

            # Current state vector (used as initial condition)
            x_solver = [T_in_true, T_m_true, W_in_true, C_in_frac]

            # Adjacent zone temperatures
            current_adj_zones = [{'T_in': self.exchange.get_variable_value(state, adj['handle_T_in']),
                                  'R_env': adj['R_env']} for adj in z.adj_zones]

            params = {
                'C_air': z.C_air, 'C_mass': z.C_mass,
                'R_env_ext': z.R_env_ext, 'R_env_gnd': z.R_env_gnd,
                'R_int': z.R_int, 'M_air': z.M_air, 'V_room': z.V_room,
                'T_out': T_out, 'N_occ': N_occ, 'Q_equip': Q_equip,
                'T_s': T_s, 'W_s': W_s, 'C_s': C_s_frac,
                'd_T': 0.0, 'd_W': 0.0, 'd_C': 0.0,
                'adj_zones': current_adj_zones
            }

            dt_hours = self.exchange.system_time_step(state) or self.exchange.zone_time_step(state)
            time_vec = [0, dt_hours * 3600.0]

            # Simulate one step
            response = ct.input_output_response(z.sys_ode, time_vec, U=[V_dot_s], X0=x_solver, params=params)
            x_pred_next = response.states[:, -1]

            # ---- Logging ----
            has_prev = (z.prev_prediction is not None)
            log_entry = {
                "timestamp": base_date,
                "T_in_actual": T_in_true,
                "T_in_pred":   z.prev_prediction[0] if has_prev else float('nan'),
                "T_m_actual":  T_m_true,
                "T_m_pred":    z.prev_prediction[1] if has_prev else float('nan'),
                "W_in_actual": W_in_true,
                "W_in_pred":   z.prev_prediction[2] if has_prev else float('nan'),
                "C_in_actual": C_in_ppm,                                # log in ppm for readability
                "C_in_pred":   z.prev_prediction[3]*1e6 if has_prev else float('nan'),  # convert back to ppm
                "T_out": T_out,
                "V_dot_s": V_dot_s
            }
            z.log.append(log_entry)

            # Store prediction for next timestep (in internal units: W_in kg/kg, C_in m³/m³)
            z.prev_prediction = x_pred_next

    except Exception as e:
        print(f"\n--- Python Exception in zone_model ---")
        print(f"Error: {e}")
        traceback.print_exc()
        print("----------------------------------------\n")

sim.zone_model        = types.MethodType(zone_model, sim)

# Register them on the EnergyPlus "begin" hook
sim.register_handlers("begin", [
    {"method_name": "zone_model"},
])

['state_logger', 'co2_set_outdoor_ppm', 'people_injector', 'zone_model']

In [12]:
# @title zone_estimate_ekf
def zone_estimate_ekf(self, state):
    """Extended Kalman Filter for all thermal zones, estimating states and disturbances."""
    try:
        if not self.exchange.api_data_fully_ready(state):
            return

        # --- One‑time setup: list of zones ---
        if not hasattr(self, 'zones_ekf'):
            self.zones_ekf = {}
            all_params = self.get_zone_thermal_parameters()
            self.ekf_zone_list = list(all_params.keys())
            print(f"[EKF] Will handle zones: {self.ekf_zone_list}")

        day   = self.exchange.day_of_year(state)
        time  = self.exchange.current_time(state)
        base_date = datetime.datetime(2026, 1, 1) + datetime.timedelta(
                        days=day-1, seconds=int(time*3600))

        # ---- Noise configuration ----
        SIMULATE_NOISE = True
        SIMULATE_SUPPLY_NOISE = False

        sigma_T = 0.1          # °C
        sigma_W = 0.0001       # kg/kg
        sigma_C_ppm = 25.0     # ppm

        # Time step (seconds)
        dt_hours = self.exchange.system_time_step(state) or self.exchange.zone_time_step(state)
        dt = dt_hours * 3600.0
        if dt <= 0:
            return

        # Constants
        rho_air, cp_air = 1.204, 1006.0
        q_person   = 100.0       # W/person
        g_w_person = 5e-5        # kg/s·person
        g_co2_person = 1e-5      # m³/s·person (volumetric CO₂ at room conditions)

        # ---- Loop over all zones ----
        for zone_id in self.ekf_zone_list:

            # ========== Initialise zone ==========
            if zone_id not in self.zones_ekf:
                raw_params = self.get_zone_thermal_parameters()[zone_id]

                # Variable handles (same as in zone_model)
                handles = {
                    "T_in": self.exchange.get_variable_handle(state, "Zone Mean Air Temperature", zone_id),
                    "T_m":  self.exchange.get_variable_handle(state, "Zone Mean Radiant Temperature", zone_id),
                    "W_in": self.exchange.get_variable_handle(state, "Zone Mean Air Humidity Ratio", zone_id),
                    "CO2_in": self.exchange.get_variable_handle(state, "Zone Air CO2 Concentration", zone_id),
                    "N_occ": self.exchange.get_variable_handle(state, "Zone People Occupant Count", zone_id),
                    "T_out": self.exchange.get_variable_handle(state, "Site Outdoor Air Drybulb Temperature", "Environment"),
                    "Q_equip": self.exchange.get_variable_handle(state, "Zone Electric Equipment Total Heating Rate", zone_id),
                    "V_dot": self.exchange.get_variable_handle(state, "System Node Current Density Volume Flow Rate", f"{zone_id} In Node"),
                    "T_s": self.exchange.get_variable_handle(state, "System Node Temperature", "VAV Sys 1 Outlet Node"),
                    "W_s": self.exchange.get_variable_handle(state, "System Node Humidity Ratio", "VAV Sys 1 Outlet Node"),
                    "C_s": self.exchange.get_variable_handle(state, "System Node CO2 Concentration", "VAV Sys 1 Outlet Node"),
                }

                # Thermal resistances
                inv_R_env_ext = 0.0
                adj_zones = []
                for b in raw_params["boundaries"]:
                    target = b["target"]
                    r_abs  = float(b["R_absolute_K_W"])
                    if target == "Ground":
                        R_env_gnd = r_abs
                    elif target == "Environment" or b["boundary_condition"] == "outdoors":
                        inv_R_env_ext += 1.0 / r_abs
                    else:
                        adj_zones.append({
                            "zone": target,
                            "R_env": r_abs,
                            "handle_T_in": self.exchange.get_variable_handle(state, "Zone Mean Air Temperature", target)
                        })
                R_env_ext = 1.0 / inv_R_env_ext if inv_R_env_ext > 0 else float('inf')

                # Precomputed sum of 1/R_adj for Jacobian
                inv_R_adj_sum = sum(1.0 / float(adj["R_env"]) for adj in adj_zones)

                # ---- EKF matrices ----
                # State: [T_in, T_m, W_in, C_in(frac), d_T, d_W, N_occ]
                P_est = np.eye(7) * 1.0
                P_est[6, 6] = 10.0   # occupancy uncertainty

                # Process noise covariance (discrete per step, units of state²)
                Q = np.diag([
                    0.1,     # T_in   (C²)
                    5.0,     # T_m    (C²)
                    1e-6,    # W_in   (kg/kg)²
                    1e-8,    # C_in   (fraction²)  ≈ (100 ppm)²
                    1.0,     # d_T    (W²)        (reduced from 50)
                    1e-5,    # d_W    (kg/s)²
                    10.0     # N_occ  (count²)
                ])

                # Measurement noise covariance (sensor noise in measurement units)
                if SIMULATE_NOISE:
                    R = np.diag([
                        sigma_T**2,
                        sigma_W**2,
                        (sigma_C_ppm * 1e-6)**2   # convert to fraction
                    ])
                else:
                    R = np.diag([0.01, 1e-8, 1e-10])

                # Observation matrix: we measure T_in, W_in, C_in(frac)
                H = np.zeros((3, 7))
                H[0, 0] = 1.0   # T_in
                H[1, 2] = 1.0   # W_in
                H[2, 3] = 1.0   # C_in

                self.zones_ekf[zone_id] = types.SimpleNamespace(
                    V_room    = float(raw_params['V_room']),
                    M_air     = float(raw_params['M_air']),
                    C_air     = float(raw_params['C_air']),
                    C_mass    = float(raw_params['C_mass']),
                    R_int     = float(raw_params['R_int']),
                    R_env_gnd = R_env_gnd if 'R_env_gnd' in locals() else float('inf'),
                    R_env_ext = R_env_ext,
                    inv_R_adj_sum = inv_R_adj_sum,
                    adj_zones = adj_zones,
                    handles   = handles,
                    X_est     = None,
                    P_est     = P_est,
                    Q         = Q,
                    R         = R,
                    H         = H,
                    log       = [],
                )
                print(f"[EKF] Initialised for {zone_id}")

            # ========== Runtime EKF step ==========
            z = self.zones_ekf[zone_id]

            # ---- 1. Measurements & inputs ----
            t_in_meas = self.exchange.get_variable_value(state, z.handles["T_in"])
            w_in_meas = self.exchange.get_variable_value(state, z.handles["W_in"])
            c_in_ppm  = self.exchange.get_variable_value(state, z.handles["CO2_in"])   # ppm

            V_dot_s = self.exchange.get_variable_value(state, z.handles["V_dot"])
            T_out   = self.exchange.get_variable_value(state, z.handles["T_out"])
            Q_equip = 0.0   # unmeasured – d_T will absorb it
            T_s     = self.exchange.get_variable_value(state, z.handles["T_s"])
            W_s     = self.exchange.get_variable_value(state, z.handles["W_s"])
            C_s_ppm = self.exchange.get_variable_value(state, z.handles["C_s"])       # ppm

            # Add sensor noise
            if SIMULATE_NOISE:
                t_in_meas += np.random.normal(0, sigma_T)
                w_in_meas += np.random.normal(0, sigma_W)
                c_in_ppm  += np.random.normal(0, sigma_C_ppm)
            if SIMULATE_SUPPLY_NOISE:
                T_s += np.random.normal(0, sigma_T)
                W_s += np.random.normal(0, sigma_W)
                C_s_ppm += np.random.normal(0, sigma_C_ppm)

            # ---- 2. Unit conversion: ppm → volumetric fraction ----
            c_in_frac = c_in_ppm * 1e-6
            C_s_frac  = C_s_ppm  * 1e-6

            # Ground truth (only for logging)
            t_m_true  = self.exchange.get_variable_value(state, z.handles["T_m"])
            n_occ_true = self.exchange.get_variable_value(state, z.handles["N_occ"])

            # ----- First call: initialise state estimate -----
            if z.X_est is None:
                z.X_est = np.array([t_in_meas, t_in_meas, w_in_meas, c_in_frac,
                                    0.0, 0.0, 0.0])
                z.last_time = (day * 24.0) + time   # not used but kept for consistency
                continue   # skip prediction on first step

            # ----- Current state estimate -----
            T_in_e, T_m_e, W_in_e, C_in_e, d_T_e, d_W_e, N_occ_e = z.X_est

            # ----- 3. EKF Predict (state) -----
            # Adjacent zone heat flux
            q_adj_sum = 0.0
            inv_R_adj = 0.0
            for adj in z.adj_zones:
                t_adj = self.exchange.get_variable_value(state, adj["handle_T_in"])
                q_adj_sum += t_adj / float(adj["R_env"])
                inv_R_adj += 1.0 / float(adj["R_env"])
            # Ground conduction is omitted (unknown)
            q_env  = (T_out - T_in_e) / z.R_env_ext if z.R_env_ext < float('inf') else 0.0
            q_mass = (T_m_e - T_in_e) / z.R_int
            q_int  = N_occ_e * q_person + Q_equip
            q_s    = rho_air * V_dot_s * cp_air * (T_s - T_in_e)

            dT_in_dt = (q_env + q_adj_sum - T_in_e*inv_R_adj + q_mass + q_int + q_s + d_T_e) / z.C_air
            dT_m_dt  = (T_in_e - T_m_e) / (z.C_mass * z.R_int)
            dot_m_s  = rho_air * V_dot_s
            dW_in_dt = (N_occ_e * g_w_person + dot_m_s * (W_s - W_in_e) + d_W_e) / z.M_air
            dC_in_dt = (N_occ_e * g_co2_person + V_dot_s * (C_s_frac - C_in_e)) / z.V_room

            # Euler integration for state prediction
            X_pred = z.X_est + np.array([dT_in_dt, dT_m_dt, dW_in_dt, dC_in_dt, 0.0, 0.0, 0.0]) * dt

            # ----- 4. EKF Predict (covariance) -----
            # Jacobian of the continuous dynamics
            df_dX = np.zeros((7, 7))
            inv_R_ext = 1.0 / z.R_env_ext if z.R_env_ext < float('inf') else 0.0
            inv_R_int = 1.0 / z.R_int if z.R_int < float('inf') else 0.0

            df_dX[0, 0] = (-inv_R_ext - inv_R_adj - inv_R_int - (rho_air * cp_air * V_dot_s)) / z.C_air
            df_dX[0, 1] = inv_R_int / z.C_air
            df_dX[0, 4] = 1.0 / z.C_air
            df_dX[0, 6] = q_person / z.C_air

            df_dX[1, 0] = inv_R_int / z.C_mass
            df_dX[1, 1] = -inv_R_int / z.C_mass

            df_dX[2, 2] = -(rho_air * V_dot_s) / z.M_air
            df_dX[2, 5] = 1.0 / z.M_air
            df_dX[2, 6] = g_w_person / z.M_air

            df_dX[3, 3] = -V_dot_s / z.V_room
            df_dX[3, 6] = g_co2_person / z.V_room

            # Discrete‑time transition (Euler)
            F = np.eye(7) + df_dX * dt
            P_pred = F @ z.P_est @ F.T + z.Q

            # ----- 5. EKF Update -----
            # Measurement vector (same units as H)
            Y_meas = np.array([t_in_meas, w_in_meas, c_in_frac])
            y = Y_meas - (z.H @ X_pred)          # innovation
            S = z.H @ P_pred @ z.H.T + z.R       # innovation covariance
            K = P_pred @ z.H.T @ np.linalg.inv(S) # Kalman gain

            # Joseph form for covariance update (ensures positive definiteness)
            IKH = np.eye(7) - K @ z.H
            z.P_est = IKH @ P_pred @ IKH.T + K @ z.R @ K.T
            z.X_est = X_pred + K @ y

            # Ensure physically plausible values
            z.X_est[2] = max(0.0, z.X_est[2])   # W_in ≥ 0
            z.X_est[3] = max(0.0, z.X_est[3])   # C_in ≥ 0
            z.X_est[6] = max(0.0, z.X_est[6])   # N_occ ≥ 0

            # ----- 6. Logging (in human‑readable units) -----
            log_entry = {
                "timestamp": base_date,
                "T_in_actual": t_in_meas,
                "T_in_pred":   z.X_est[0],
                "T_m_actual":  t_m_true,
                "T_m_pred":    z.X_est[1],
                "W_in_actual": w_in_meas,
                "W_in_pred":   z.X_est[2],
                "C_in_actual": c_in_ppm,       # ppm for readability
                "C_in_pred":   z.X_est[3] * 1e6,  # convert fraction → ppm
                "N_occ_actual": n_occ_true,
                "N_occ_est":    z.X_est[6],
                "d_T_est":      z.X_est[4],
                "d_W_est":      z.X_est[5],
                "T_out": T_out,
                "V_dot_s": V_dot_s
            }
            z.log.append(log_entry)

    except Exception as e:
        print(f"\n--- Python Exception in zone_estimate_ekf ---")
        print(f"Error: {e}")
        traceback.print_exc()
        print("----------------------------------------\n")
# Attach the functions as methods of the simulation object
sim.zone_estimate_ekf = types.MethodType(zone_estimate_ekf, sim)

# Register them on the EnergyPlus "begin" hook
sim.register_handlers("begin", [
    {"method_name": "zone_estimate_ekf"}
])

['state_logger',
 'co2_set_outdoor_ppm',
 'people_injector',
 'zone_model',
 'zone_estimate_ekf']

In [13]:
# @title zone_mpc
def zone_mpc(self, state):
    """Multi-zone Model Predictive Control (logs stored in self.mpc_logs)."""
    if not self.exchange.api_data_fully_ready(state) or self.exchange.warmup_flag(state):
        return

    T_ref = 16.0   # °C

    # 1. Retrieve all active EKF zones
    if not hasattr(self, 'zones_ekf') or not self.zones_ekf:
        return

    # Ensure the MPC log container exists
    if not hasattr(self, 'mpc_logs'):
        self.mpc_logs = {}
        all_params = self.get_zone_thermal_parameters()
        self.ekf_zone_list = list(all_params.keys())
        print(f"[MPC] Will handle zones: {self.ekf_zone_list}")

    if not hasattr(self, 'vav_targets'):
        self.vav_targets = {}

    day   = self.exchange.day_of_year(state)
    time  = self.exchange.current_time(state)
    dt_hr = self.exchange.system_time_step(state) or self.exchange.zone_time_step(state)
    dt    = dt_hr * 3600.0           # seconds
    if dt <= 0:
        return

    # Physical constants
    rho_air, cp_air = 1.204, 1006.0
    q_person   = 100.0               # W/person
    g_w_person = 5e-5                # kg/s·person
    g_co2_person = 1e-5              # m³ CO₂/s·person (at room conditions)

    # ASHRAE‑based comfort limits (internal units)
    CO2_max_frac = 1000 * 1e-6       # 1000 ppm → volumetric fraction
    W_max_kgkg   = 0.012             # approx. 70 % RH at 22 °C

    # Actuator limits
    u_max = 0.5                      # realistic VAV maximum (m³/s)
    delta_u_max = 0.1                # max change per 5‑min step (m³/s)

    # MPC horizon and weights
    Np, nx, nu = 10, 4, 1
    Q_diag = [100.0, 1.0, 1e-4, 1e-4]   # T_in, T_m, C_in, W_in
    R_val = 0.1                          # penalise large u deviations
    R_delta_val = 1.0                    # slew rate penalty
    rho_c_soft = 1e6                     # slack penalty for CO₂
    rho_w_soft = 1e6                     # slack penalty for humidity

    # Loop over all zones with an EKF
    for zone_id, z_ekf in self.zones_ekf.items():
        if z_ekf.X_est is None:
            continue

        # ----- Extract EKF estimates (all in SI units) -----
        T_in_e, T_m_e, W_in_e, C_in_frac, d_T_e, d_W_e, N_occ_e = z_ekf.X_est
        x_k = np.array([T_in_e, T_m_e, C_in_frac, W_in_e])

        # ----- Boundary conditions (convert supply CO₂ from ppm → fraction) -----
        T_out = self.exchange.get_variable_value(state, z_ekf.handles["T_out"])
        T_s   = self.exchange.get_variable_value(state, z_ekf.handles["T_s"])
        W_s   = self.exchange.get_variable_value(state, z_ekf.handles["W_s"])
        C_s_ppm = self.exchange.get_variable_value(state, z_ekf.handles["C_s"])
        C_s_frac = C_s_ppm * 1e-6

        # Previous control input (for re‑linearisation and slew rate)
        u_op = getattr(z_ekf, 'u_prev', 0.05)

        # ----- Linearisation (continuous state matrix A_c) -----
        inv_R_ext = 1.0 / z_ekf.R_env_ext if z_ekf.R_env_ext > 0 else 0.0
        inv_R_int = 1.0 / z_ekf.R_int if z_ekf.R_int > 0 else 0.0
        inv_R_adj_total = getattr(z_ekf, 'inv_R_adj_sum', 0.0)

        Ac = np.zeros((4, 4))
        Ac[0, 0] = (-inv_R_ext - inv_R_int - inv_R_adj_total - (rho_air * cp_air * u_op)) / z_ekf.C_air
        Ac[0, 1] = inv_R_int / z_ekf.C_air
        Ac[1, 0] = inv_R_int / z_ekf.C_mass
        Ac[1, 1] = -inv_R_int / z_ekf.C_mass
        Ac[2, 2] = -u_op / z_ekf.V_room
        Ac[3, 3] = -(rho_air * u_op) / z_ekf.M_air

        Bc = np.zeros((4, 1))
        # Guard against zero temperature difference
        delta_T = T_s - T_in_e
        if abs(delta_T) < 0.5:   # too small → uncontrollable, keep previous
            u_opt = u_op
            m_dot_opt = float(u_opt * rho_air)
            h_act = self.exchange.get_actuator_handle(state, "System Node Setpoint",
                         "Mass Flow Rate Setpoint", f"{zone_id} In Node")
            if h_act != -1:
                self.exchange.set_actuator_value(state, h_act, m_dot_opt)

            # Log to new container
            if zone_id not in self.mpc_logs:
                self.mpc_logs[zone_id] = []
            self.mpc_logs[zone_id].append({
                "day": day, "hour": time, "T_in_est": T_in_e,
                "T_m_est": T_m_e, "W_in_est": W_in_e, "C_in_est": C_in_frac*1e6,
                "N_occ_est": N_occ_e, "T_ref": T_ref, "T_ss": T_in_e,
                "u_ss": u_op, "u_opt_m3s": u_opt
            })
            continue


        Bc[0, 0] = (rho_air * cp_air * delta_T) / z_ekf.C_air
        Bc[2, 0] = (C_s_frac - C_in_frac) / z_ekf.V_room
        Bc[3, 0] = (rho_air * (W_s - W_in_e)) / z_ekf.M_air

        # ----- Drift vector c_c (continuous) -----
        t_adj_flux = 0.0
        for adj in z_ekf.adj_zones:
            t_adj = self.exchange.get_variable_value(state, adj["handle_T_in"])
            t_adj_flux += t_adj / float(adj["R_env"])

        f_op = np.zeros((4, 1))
        f_op[0, 0] = ((T_out - T_in_e) * inv_R_ext +
                      (T_m_e - T_in_e) * inv_R_int +
                      (t_adj_flux - T_in_e * inv_R_adj_total) +
                      (rho_air * u_op * cp_air * (T_s - T_in_e)) +
                      (N_occ_e * q_person) + d_T_e) / z_ekf.C_air
        f_op[1, 0] = ((T_in_e - T_m_e) * inv_R_int) / z_ekf.C_mass
        f_op[2, 0] = (u_op * (C_s_frac - C_in_frac) + N_occ_e * g_co2_person) / z_ekf.V_room
        f_op[3, 0] = (rho_air * u_op * (W_s - W_in_e) + d_W_e) / z_ekf.M_air

        cc = f_op - np.dot(Ac, x_k.reshape(4,1)) - (Bc * u_op)

        # ----- Exact discretisation (augmented matrix exponential) -----
        M_aug = np.zeros((6, 6))
        M_aug[0:4, 0:4] = Ac
        M_aug[0:4, 4:5] = Bc
        M_aug[0:4, 5:6] = cc
        Md = la.expm(M_aug * dt)

        Ad = Md[0:4, 0:4]
        Bd = Md[0:4, 4:5]
        cd = Md[0:4, 5:6].flatten()   # affine term in discrete time

        # ----- Target selector (thermal subsystem only) -----

        Ad_th = Ad[0:2, 0:2]
        Bd_th = Bd[0:2, :]
        cd_th = cd[0:2]

        top_block = np.hstack([np.eye(2) - Ad_th, -Bd_th])
        bottom_block = np.array([[1.0, 0.0, 0.0]])   # track T_in
        M_ss = np.vstack([top_block, bottom_block])
        rhs_ss = np.concatenate([cd_th, [T_ref]])

        try:
            ss_res = np.linalg.solve(M_ss, rhs_ss)
            x_ss = np.array([ss_res[0], ss_res[1], C_in_frac, W_in_e])   # hold CO₂/W at current
            u_ss = ss_res[2]
        except np.linalg.LinAlgError:
            # Fallback: hold current state and previous control
            x_ss = x_k.copy()
            u_ss = u_op

        # ----- Build OSQP problem (absolute variables, slacks) -----
        # Decision vector: z = [x0, ..., x_Np, u0, ..., u_{Np-1}, eps_c0, ..., eps_w{Np-1}]
        # Total length: (Np+1)*nx + Np*nu + Np*2
        n_states = (Np+1)*nx + Np*nu + Np*2

        # Hessian
        Q = sparse.diags(Q_diag, format='csc')
        R = sparse.diags([R_val], format='csc')
        P = sparse.block_diag([
            sparse.kron(sparse.eye(Np+1), Q),
            sparse.kron(sparse.eye(Np), R),
            sparse.diags([rho_c_soft]*Np + [rho_w_soft]*Np)
        ], format='csc')

        # Linear gradient (to shift cost towards x_ss, u_ss, zero slacks)
        z_ss = np.hstack([
            np.tile(x_ss, Np+1),
            np.tile(u_ss, Np),
            np.zeros(Np),   # epsilon_c target = 0
            np.zeros(Np)    # epsilon_w target = 0
        ])
        q = -P @ z_ss

        # Constraints
        Ax_dyn = sparse.kron(sparse.eye(Np+1, Np+1), sparse.eye(nx)) - \
                 sparse.kron(sparse.diags([1]*Np, offsets=[-1], shape=(Np+1, Np+1)), Ad)
        Bu_dyn = sparse.kron(sparse.vstack([
            sparse.csc_matrix((1, Np)),
            sparse.eye(Np)
        ]), -Bd)
        A_eq = sparse.hstack([Ax_dyn, Bu_dyn, sparse.csc_matrix((Np*nx, 2*Np))], format='csc')
        l_eq = np.tile(cd, Np)
        u_eq = l_eq

        # Initial condition
        A0 = sparse.hstack([sparse.eye(nx),
                             sparse.csc_matrix((nx, Np*nx + Np*nu + 2*Np))], format='csc')
        A_eq = sparse.vstack([A0, A_eq])
        l_eq = np.hstack([x_k, l_eq])
        u_eq = np.hstack([x_k, u_eq])

        # Actuator limits
        A_u = sparse.hstack([
            sparse.csc_matrix((Np*nu, (Np+1)*nx)),
            sparse.eye(Np*nu),
            sparse.csc_matrix((Np*nu, 2*Np))
        ], format='csc')
        l_u = np.zeros(Np*nu)
        u_u = np.ones(Np*nu) * u_max

        # Slew-rate limits
        A_slew = sparse.hstack([
            sparse.csc_matrix((Np, (Np+1)*nx)),
            sparse.diags([-1, 1], offsets=[-1, 0], shape=(Np, Np)),
            sparse.csc_matrix((Np, 2*Np))
        ], format='csc')
        l_slew = np.ones(Np) * (-delta_u_max)
        u_slew = np.ones(Np) * ( delta_u_max)
        l_slew[0] += u_op
        u_slew[0] += u_op

        # Soft constraints on CO₂ and humidity
        A_co2_full = sparse.lil_matrix((Np, n_states))
        for i in range(Np):
            A_co2_full[i, i*nx + 2] = 1.0
        A_co2_full[:, (Np+1)*nx + Np*nu : (Np+1)*nx + Np*nu + Np] = -sparse.eye(Np)
        A_co2_full = A_co2_full.tocsc()

        A_w = sparse.lil_matrix((Np, n_states))
        for i in range(Np):
            A_w[i, i*nx + 3] = 1.0
        A_w[:, (Np+1)*nx + Np*nu + Np : (Np+1)*nx + Np*nu + 2*Np] = -sparse.eye(Np)
        A_w = A_w.tocsc()

        A_ineq = sparse.vstack([
            A_u,
            A_slew,
            A_co2_full,
            A_w
        ], format='csc')

        l_ineq = np.hstack([
            l_u,
            l_slew,
            np.ones(Np) * (-np.inf),
            np.ones(Np) * (-np.inf)
        ])
        u_ineq = np.hstack([
            u_u,
            u_slew,
            np.ones(Np) * CO2_max_frac,
            np.ones(Np) * W_max_kgkg
        ])

        A_slack_nonneg = sparse.hstack([
            sparse.csc_matrix((2*Np, (Np+1)*nx + Np*nu)),
            sparse.eye(2*Np)
        ])
        A_ineq = sparse.vstack([A_ineq, A_slack_nonneg], format='csc')
        l_ineq = np.hstack([l_ineq, np.zeros(2*Np)])
        u_ineq = np.hstack([u_ineq, np.ones(2*Np) * np.inf])

        # Add slew-rate penalty to Hessian
        D = sparse.eye(Np) - sparse.diags([1]* (Np-1), offsets=[-1], shape=(Np, Np))
        R_delta_mat = sparse.diags([R_delta_val]*Np, format='csc')
        P_blocks = [
            sparse.kron(sparse.eye(Np+1), Q),
            sparse.kron(sparse.eye(Np), R) + D.T @ R_delta_mat @ D,
            sparse.diags([rho_c_soft]*Np + [rho_w_soft]*Np)
        ]
        P = sparse.block_diag(P_blocks, format='csc')
        q = -P @ z_ss

        # ----- OSQP Setup / Update -----
        mpc_attr = f"mpc_{zone_id}"
        if not hasattr(self, mpc_attr):
            setattr(self, mpc_attr, {
                "prob": osqp.OSQP(),
                "initialized": False
            })
        mpc_data = getattr(self, mpc_attr)

        if not mpc_data["initialized"]:
            mpc_data["prob"].setup(P, q,
                                   sparse.vstack([A_eq, A_ineq]),
                                   np.hstack([l_eq, l_ineq]),
                                   np.hstack([u_eq, u_ineq]),
                                   verbose=False, warm_start=True)
            mpc_data["initialized"] = True
        else:
            mpc_data["prob"].update(q=q, l=np.hstack([l_eq, l_ineq]),
                                    u=np.hstack([u_eq, u_ineq]),
                                    Px=None)

        res = mpc_data["prob"].solve()

        if res.info.status_val == 1:
            u_opt = res.x[(Np+1)*nx]
        else:
            print(f"[MPC {zone_id}] Solver issue: {res.info.status}. Using u_ss.")
            u_opt = u_ss

        u_opt = np.clip(u_opt, 0.0, u_max)
        z_ekf.u_prev = u_opt

        # ----- Actuation -----
        m_dot_opt = float(u_opt * rho_air)
        self.vav_targets[zone_id] = m_dot_opt


        # ----- Logging (to separate container) -----
        if zone_id not in self.mpc_logs:
                self.mpc_logs[zone_id] = []
        self.mpc_logs[zone_id].append({
            "day": day, "hour": time, "T_in_est": T_in_e,
            "T_m_est": T_m_e, "W_in_est": W_in_e, "C_in_est": C_in_frac*1e6,
            "N_occ_est": N_occ_e, "T_ref": T_ref, "T_ss": T_in_e,
            "u_ss": u_op, "u_opt_m3s": u_opt
        })
        continue

        if int(time*60) % 60 == 0:
            print(f"[{zone_id} H:{time:.1f}] T:{T_in_e:.1f}°C T_ss:{x_ss[0]:.1f}°C "
                  f"u_ss:{u_ss:.3f} m³/s u_opt:{u_opt:.3f} m³/s "
                  f"CO2:{C_in_frac*1e6:.0f} ppm")

sim.zone_mpc = types.MethodType(zone_mpc, sim)
sim.register_handlers("before_hvac", [{"method_name": "zone_mpc"}])

['zone_mpc']

In [15]:
# @title enforce_vav_flow
def enforce_vav_flow(self, state):
    """Continuously enforces the VAV node clamps and schedules inside the HVAC iteration."""
    if not self.exchange.api_data_fully_ready(state) or self.exchange.warmup_flag(state):
        return

    # --- 1. ENFORCE SCHEDULE OVERRIDES ---
    # Expand the thermostat deadband so the default controller sleeps
    self.tick_set_actuator(state, component_type="Schedule", control_type="Schedule Value", actuator_key="Htg-SetP-Sch", value=-50.0, allow_warmup=False)
    self.tick_set_actuator(state, component_type="Schedule", control_type="Schedule Value", actuator_key="Clg-SetP-Sch", value=50.0, allow_warmup=False)

    # Force the Central AHU to output exactly 14.0 °C air
    self.tick_set_actuator(state, component_type="Schedule", control_type="Schedule Value", actuator_key="Seasonal Reset Supply Air Temp Sch", value=14.0, allow_warmup=False)

    # --- 2. ENFORCE VAV FLOW CLAMPS ---
    if not hasattr(self, 'vav_targets'):
        return

    # Loop through the targets saved by the MPC and clamp the inlet nodes
    for z_id, m_dot in self.vav_targets.items():
        node_name = f"{z_id} ATU In Node"
        m_dot = 0.15
        self.tick_set_actuator(state, component_type="System Node Setpoint", control_type="Mass Flow Rate Minimum Available Setpoint", actuator_key=node_name, value=m_dot, allow_warmup=False)
        self.tick_set_actuator(state, component_type="System Node Setpoint", control_type="Mass Flow Rate Maximum Available Setpoint", actuator_key=node_name, value=m_dot, allow_warmup=False)
        self.tick_set_actuator(state, component_type="System Node Setpoint", control_type="Mass Flow Rate Setpoint", actuator_key=node_name, value=m_dot, allow_warmup=False)

sim.enforce_vav_flow = types.MethodType(enforce_vav_flow, sim)

# Register this specifically on the "inside_iter" hook!
sim.register_handlers("inside_iter", [{"method_name": "enforce_vav_flow"}])


 # h_act = self.exchange.get_actuator_handle(state, "System Node Setpoint",
        #              "Mass Flow Rate Setpoint", f"{zone_id} In Node")
        # if h_act != -1:
        #     self.exchange.set_actuator_value(state, h_act, m_dot_opt)

        # ----- Actuation: Hard Clamp via System Nodes -----
        # node_name = f"{zone_id} In Node"

        # self.tick_set_actuator(
        #     state,
        #     component_type="System Node Setpoint",
        #     control_type="Mass Flow Rate Setpoint",
        #     actuator_key=node_name,
        #     value=m_dot_opt,
        #     allow_warmup=False,
        #     when="on_change",
        #     label=f"MPC Override TARGET {zone_id}"
        # )
        # # TARGET THE INLET OF THE VAV BOX!
        # node_name = f"{zone_id} ATU In Node"

        # # 1. Clamp the Minimum Boundary UP to our optimal flow
        # self.tick_set_actuator(
        #     state,
        #     component_type="System Node Setpoint",
        #     control_type="Mass Flow Rate Minimum Available Setpoint",
        #     actuator_key=node_name,
        #     value=m_dot_opt,
        #     allow_warmup=False,
        #     when="on_change",
        #     label=f"MPC Override MIN {zone_id}"
        # )

        # # 2. Clamp the Maximum Boundary DOWN to our optimal flow
        # self.tick_set_actuator(
        #     state,
        #     component_type="System Node Setpoint",
        #     control_type="Mass Flow Rate Maximum Available Setpoint",
        #     actuator_key=node_name,
        #     value=m_dot_opt,
        #     allow_warmup=False,
        #     when="on_change",
        #     label=f"MPC Override MAX {zone_id}"
        # )

        # # 3. Set the target setpoint for good measure
        # self.tick_set_actuator(
        #     state,
        #     component_type="System Node Setpoint",
        #     control_type="Mass Flow Rate Setpoint",
        #     actuator_key=node_name,
        #     value=m_dot_opt,
        #     allow_warmup=False,
        #     when="on_change",
        #     label=f"MPC Override TARGET {zone_id}"
        # )

['enforce_vav_flow']

## Run Simualtion

In [16]:
# @title Run Simulation
# Set Simulation Time
sim.set_simulation_params(
    start=(1, 1),
    end=(1, 7),
    timestep_per_hour = 12, # 4 (every 15 minutes) or 6 (every 10 minutes).
    start_day_of_week="Sunday",
)

print("Starting EnergyPlus Uncontrolled Simulation...")
res = sim.run_annual()

if(res == 0):
    print("Simulation Complete! Converting data to Pandas...")
    # Create the DataFrame
    SimulationData = pd.DataFrame(sim.collected_data)
    cols = ['timestamp'] + [c for c in SimulationData.columns if c != 'timestamp']
    SimulationData = SimulationData[cols]
    print("Done.")

if(res == 1):
    err_path = Path(OUT_DIR) / "eplusout.err"
    if err_path.exists():
        print("--- EnergyPlus Error Log ---")
        with open(err_path, 'r') as f:
            print(f.read()[-4000:])
    else:
        print(f"Could not find the error file at: {err_path}")

Starting EnergyPlus Uncontrolled Simulation...

[Injector] Mapped 5 actuators across 5 zones (Extrapolation Active).
[Zone Model] Active zones: ['PLENUM-1', 'SPACE1-1', 'SPACE2-1', 'SPACE3-1', 'SPACE4-1', 'SPACE5-1']
[PLENUM-1] RC model ready. Adjacent: ['SPACE1-1', 'SPACE2-1', 'SPACE3-1', 'SPACE4-1', 'SPACE5-1']
[SPACE1-1] RC model ready. Adjacent: ['PLENUM-1', 'SPACE2-1', 'SPACE4-1', 'SPACE5-1']
[SPACE2-1] RC model ready. Adjacent: ['PLENUM-1', 'SPACE1-1', 'SPACE3-1', 'SPACE5-1']
[SPACE3-1] RC model ready. Adjacent: ['PLENUM-1', 'SPACE2-1', 'SPACE4-1', 'SPACE5-1']
[SPACE4-1] RC model ready. Adjacent: ['PLENUM-1', 'SPACE1-1', 'SPACE3-1', 'SPACE5-1']
[SPACE5-1] RC model ready. Adjacent: ['PLENUM-1', 'SPACE1-1', 'SPACE2-1', 'SPACE3-1', 'SPACE4-1']
[EKF] Will handle zones: ['PLENUM-1', 'SPACE1-1', 'SPACE2-1', 'SPACE3-1', 'SPACE4-1', 'SPACE5-1']
[EKF] Initialised for PLENUM-1
[EKF] Initialised for SPACE1-1
[EKF] Initialised for SPACE2-1
[EKF] Initialised for SPACE3-1
[EKF] Initialised for

## Results

In [17]:
# @title plot_zone_dashboard
def plot_zone_dashboard(zone_id,
                        raw_df=None,       # from sim.collected_data
                        ekf_df=None,       # from sim.zones_ekf[zone_id].log
                        mpc_df=None,       # from sim.zones_ekf[zone_id].mpc_logs
                        comfort_limits=None):  # optional dict to override defaults
    """
    Creates a unified dashboard for a single thermal zone.

    Subplots include:
      - Temperature (actual, EKF, supply, outdoor, reference)
      - Humidity ratio (actual, EKF, supply) with comfort band
      - CO₂ concentration (actual, EKF, supply) with comfort limit
      - Occupancy (actual people count vs EKF estimate)
      - Estimated disturbances (thermal & moisture)
      - Control airflow (optimal, steady‑state, and actual flow if available)

    After plotting, a validation table is printed.

    Parameters
    ----------
    zone_id : str
        Zone identifier, e.g. "SPACE1-1"
    raw_df : pandas.DataFrame, optional
        DataFrame from sim.collected_data (raw EnergyPlus states).
        Must have a datetime index.
    ekf_df : pandas.DataFrame, optional
        DataFrame from sim.zones_ekf[zone_id].log (EKF outputs)
    mpc_df : pandas.DataFrame, optional
        DataFrame from sim.zones_ekf[zone_id].mpc_logs (MPC outputs)
    comfort_limits : dict, optional
        Override defaults, e.g. {"W_max": 0.014, "W_min": 0.002, "CO2_max_ppm": 1200}
    """
    # ---------- comfort bounds ----------
    limits = {
        "W_max": 0.012,        # kg/kg   ~70% RH at 22°C
        "W_min": 0.004,        # kg/kg   ~30% RH at 22°C
        "CO2_max_ppm": 1000,   # ppm
    }
    if comfort_limits is not None:
        limits.update(comfort_limits)

    # ---------- figure layout ----------
    rows = 7   # we will create all rows, some may be hidden if data missing
    titles = [
        f"[{zone_id}] Temperature Dynamics",
        f"[{zone_id}] Humidity Ratio (kg/kg)",
        f"[{zone_id}] CO₂ Concentration (ppm)",
        f"[{zone_id}] Occupancy",
        f"[{zone_id}] Disturbances (d_T, d_W)",
        f"[{zone_id}] Control Airflow (m³/s)",
        "Error Summary Table"
    ]
    specs = [
        [{"secondary_y": False}],  # Temp
        [{"secondary_y": False}],  # Hum
        [{"secondary_y": False}],  # CO2
        [{"secondary_y": False}],  # Occ
        [{"secondary_y": True}],   # Dist (two y‑axes)
        [{"secondary_y": False}],  # Flow
        [{"secondary_y": False}],  # table (will be made invisible)
    ]

    fig = make_subplots(
        rows=rows, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.04,
        subplot_titles=titles,
        specs=specs
    )

    # ---------- helper to add a trace safely ----------
    def add(col, name, row, col_idx=1, dash='solid', color=None, secondary_y=False):
        if col is not None and not col.dropna().empty:
            fig.add_trace(
                go.Scatter(x=col.index, y=col, name=name,
                           line=dict(color=color, dash=dash)),
                row=row, col=col_idx, secondary_y=secondary_y
            )

    # ---------- prepare data columns ----------
    # We'll create Series for each quantity, using the best available data
    # index: we'll use the raw_df index if available, else ekf, else mpc
    if raw_df is not None:
        idx = raw_df.index
    elif ekf_df is not None:
        idx = ekf_df.index
    elif mpc_df is not None:
        idx = mpc_df.index
    else:
        raise ValueError("At least one DataFrame must be provided.")

    # We'll extract columns; if a DataFrame is missing, the corresponding series will be None
    def get_col(df, col_name):
        if df is not None and col_name in df.columns:
            return df[col_name].dropna()
        return None

    # Temperature columns
    T_in_actual = get_col(raw_df, f"{zone_id}_T_in")
    T_in_est    = get_col(ekf_df, "T_in_pred")   # EKF logs as "T_in_pred"
    T_m_actual  = get_col(raw_df, f"{zone_id}_T_m")
    T_m_est     = get_col(ekf_df, "T_m_pred")
    T_s         = get_col(raw_df, "T_s")
    T_out       = get_col(raw_df, "T_out")
    T_ref       = get_col(mpc_df, "T_ref")   # from MPC logs

    # Humidity
    W_in_actual = get_col(raw_df, f"{zone_id}_W_in")
    W_in_est    = get_col(ekf_df, "W_in_pred")
    W_s         = get_col(raw_df, "W_s")

    # CO2 (all in ppm)
    C_in_actual = get_col(raw_df, f"{zone_id}_CO2_in")
    C_in_est    = get_col(ekf_df, "C_in_pred")  # EKF logs now back to ppm
    C_s         = get_col(raw_df, "C_s")

    # Occupancy
    N_occ_actual = get_col(raw_df, f"{zone_id}_Occ")
    N_occ_est    = get_col(ekf_df, "N_occ_est")

    # Disturbances
    d_T_est = get_col(ekf_df, "d_T_est")
    d_W_est = get_col(ekf_df, "d_W_est")

    # Control flows
    u_opt  = get_col(mpc_df, "u_opt_m3s")
    u_ss   = get_col(mpc_df, "u_ss")
    V_dot_actual = get_col(raw_df, f"{zone_id}_V_dot")  # actual flow from E+

    # ---------- plot rows ----------
    # Row 1 – Temperature
    add(T_in_actual, 'T_in actual', row=1, color='#1f77b4')
    add(T_in_est,    'T_in EKF',     row=1, dash='dash', color='#1f77b4')
    add(T_m_actual,  'T_m actual',   row=1, color='#d62728')
    add(T_m_est,     'T_m EKF',      row=1, dash='dash', color='#d62728')
    add(T_s,         'T_supply',     row=1, color='#2ca02c')
    add(T_out,       'T_outdoor',    row=1, dash='dot', color='#ff7f0e')
    add(T_ref,       'T_ref (setpoint)', row=1, dash='dot', color='#00ff00')

    # Row 2 – Humidity
    add(W_in_actual, 'W_in actual', row=2, color='#1f77b4')
    add(W_in_est,    'W_in EKF',    row=2, dash='dash', color='#1f77b4')
    add(W_s,         'W_supply',    row=2, color='#2ca02c')
    # comfort band – two horizontal lines across the subplot
    if not (W_in_actual is None and W_in_est is None and W_s is None):
        fig.add_hline(y=limits["W_max"], line_dash="dot", line_color="red",
                      annotation_text="W_max comfort", row=2, col=1)
        fig.add_hline(y=limits["W_min"], line_dash="dot", line_color="red",
                      annotation_text="W_min comfort", row=2, col=1)

    # Row 3 – CO2
    add(C_in_actual, 'CO₂ actual', row=3, color='#1f77b4')
    add(C_in_est,    'CO₂ EKF',    row=3, dash='dash', color='#1f77b4')
    add(C_s,         'CO₂ supply', row=3, color='#2ca02c')
    fig.add_hline(y=limits["CO2_max_ppm"], line_dash="dot", line_color="red",
                  annotation_text="Max CO₂ (1000 ppm)", row=3, col=1)

    # Row 4 – Occupancy
    add(N_occ_actual, 'People actual', row=4, color='#1f77b4')
    add(N_occ_est,    'People EKF',    row=4, dash='dash', color='#1f77b4')

    # Row 5 – Disturbances (dual axes)
    add(d_T_est, 'd_T est (W)', row=5, color='#e377c2')
    if d_W_est is not None:
        fig.add_trace(
            go.Scatter(x=d_W_est.index, y=d_W_est, name='d_W est (kg/s)',
                       line=dict(color='#8c564b', dash='dot')),
            row=5, col=1, secondary_y=True
        )

    # Row 6 – Airflow control
    add(u_opt,  'u_opt (optimal)', row=6, color='#00cc96')
    add(u_ss,   'u_ss (steady)',   row=6, dash='dash', color='#bcbd22')
    add(V_dot_actual, 'V_dot actual (E+)', row=6, dash='dot', color='#9467bd')


    # ---------- layout & axes ----------
    fig.update_yaxes(title_text="Temperature (°C)", row=1, col=1)
    fig.update_yaxes(title_text="Humidity (kg/kg)", row=2, col=1)
    fig.update_yaxes(title_text="CO₂ (ppm)",         row=3, col=1)
    fig.update_yaxes(title_text="People",            row=4, col=1)
    fig.update_yaxes(title_text="d_T (W)",           row=5, col=1, secondary_y=False)
    fig.update_yaxes(title_text="d_W (kg/s)",        row=5, col=1, secondary_y=True)
    fig.update_yaxes(title_text="Flow (m³/s)",       row=6, col=1)
    fig.update_xaxes(title_text="Time", row=rows, col=1)

    fig.update_layout(
        height=300 * rows,
        title_text=f"MPC Validation Dashboard – {zone_id}",
        hovermode="x unified",
        template="plotly_dark",
        showlegend=True
    )

    fig.show()

    # ---------- error summary table ----------
    print("\n" + "="*80)
    print(f"{'VALIDATION ERROR TABLE':^80}")
    print("="*80)

    metrics = []
    def add_metric(name, actual_ser, ref_ser, unit):
        if actual_ser is None or ref_ser is None:
            return
        err = (actual_ser - ref_ser).dropna()
        mae = err.abs().mean()
        rmse = np.sqrt((err**2).mean())
        max_err = err.abs().max()
        metrics.append([name, unit, f"{mae:.3f}", f"{rmse:.3f}", f"{max_err:.3f}"])

    # Temperature tracking (actual vs setpoint)
    if T_in_actual is not None and T_ref is not None:
        add_metric("T_in track (actual vs ref)", T_in_actual, T_ref, "°C")
    # EKF T_in error (actual vs EKF)
    if T_in_actual is not None and T_in_est is not None:
        add_metric("T_in EKF error", T_in_actual, T_in_est, "°C")
    # EKF T_m error
    if T_m_actual is not None and T_m_est is not None:
        add_metric("T_m EKF error", T_m_actual, T_m_est, "°C")
    # Occupancy estimate
    if N_occ_actual is not None and N_occ_est is not None:
        add_metric("Occ. estimate", N_occ_actual, N_occ_est, "people")
    # CO2 constraint violation (actual - 1000 ppm)
    if C_in_actual is not None:
        ex = (C_in_actual - limits["CO2_max_ppm"]).clip(lower=0)
        if not ex.empty:
            metrics.append(["CO2 exceedance (avg)", "ppm",
                            f"{ex.mean():.1f}", "-", f"{ex.max():.1f}"])
    # Humidity violation (above max)
    if W_in_actual is not None:
        ex_w = (W_in_actual - limits["W_max"]).clip(lower=0)
        if not ex_w.empty:
            metrics.append(["W exceedance (avg)", "kg/kg",
                            f"{ex_w.mean():.4f}", "-", f"{ex_w.max():.4f}"])

    if metrics:
        df_table = pd.DataFrame(metrics, columns=["Metric", "Unit", "MAE", "RMSE", "Max Error"])
        print(df_table.to_string(index=False))
    else:
        print("No comparable data available for error table.")
    print("="*80)

    return fig

In [18]:
# @title Plot Results
# Convert collected data to DataFrame
raw_df = pd.DataFrame(sim.collected_data)
sim_start = pd.Timestamp("2026-01-01 00:00:00")
raw_df['datetime'] = sim_start + pd.to_timedelta(raw_df['day']-1, unit='D') + pd.to_timedelta(raw_df['hour'] + raw_df['minute']/60, unit='h')
raw_df.set_index('datetime', inplace=True)

# EKF logs for a specific zone, e.g. SPACE1-1
zone_id = "SPACE1-1"
ekf_logs = sim.zones_ekf[zone_id].log
ekf_df = pd.DataFrame(ekf_logs)
ekf_df['datetime'] = pd.to_datetime(ekf_df['timestamp'])
ekf_df.set_index('datetime', inplace=True)

# MPC logs
mpc_logs = sim.mpc_logs[zone_id]   # list of dicts
mpc_df = pd.DataFrame(mpc_logs)
mpc_df['datetime'] = sim_start + pd.to_timedelta(mpc_df['day']-1, unit='D') + pd.to_timedelta(mpc_df['hour'], unit='h')
mpc_df.set_index('datetime', inplace=True)

# Plot the dashboard
fig = plot_zone_dashboard(zone_id, raw_df=raw_df, ekf_df=ekf_df,mpc_df=mpc_df, comfort_limits={"W_max": 0.014, "W_min": 0.002, "CO2_max_ppm": 1200})


                             VALIDATION ERROR TABLE                             
                    Metric   Unit    MAE  RMSE Max Error
T_in track (actual vs ref)     °C  7.006 7.070     9.680
            T_in EKF error     °C  0.080 0.101     0.350
             T_m EKF error     °C  0.852 0.954     2.310
             Occ. estimate people  1.335 1.521     3.000
      CO2 exceedance (avg)    ppm  147.2     -    1010.1
        W exceedance (avg)  kg/kg 0.0018     -    0.0086


In [ ]:
# @title
val= sim.runtime_get_actuator(
    sim,
    component_type="AirTerminal:SingleDuct:VAV:Reheat",
    control_type="Mass Flow Rate",
    actuator_key="SPACE1-1 VAV Reheat",
    allow_warmup=False,
    default=None
)
print(val)

In [71]:
# @title
sim.list_available_actuators()

,Kind,ComponentType,ControlType,ActuatorKey,Units
0,Actuator,Schedule:Compact,Schedule Value,OCCUPY-1,[ ]
1,Actuator,Schedule:Compact,Schedule Value,LIGHTS-1,[ ]
2,Actuator,Schedule:Compact,Schedule Value,EQUIP-1,[ ]
3,Actuator,Schedule:Compact,Schedule Value,INFIL-SCH,[ ]
4,Actuator,Schedule:Compact,Schedule Value,ACTSCHD,[ ]
...,...,...,...,...,...
1209,Actuator,Outdoor Air System Node,Drybulb Temperature,CENTRAL CHILLER CONDENSER INLET NODE,[C]
1210,Actuator,Outdoor Air System Node,Wetbulb Temperature,CENTRAL CHILLER CONDENSER INLET NODE,[C]
1211,Actuator,Outdoor Air System Node,Wind Speed,CENTRAL CHILLER CONDENSER INLET NODE,[m/s]
1212,Actuator,Outdoor Air System Node,Wind Direction,CENTRAL CHILLER CONDENSER INLET NODE,[degree]


# NEW

In [19]:
# -*- coding: utf-8 -*-
"""
Fixed Airflow Control for 5ZoneAirCooled
=========================================
Stripped-down script: no MPC, no EKF, no RC model.
Just proves we can set VAV airflow to a fixed value and observe it.

FIXED_FLOW_M3S  - the volumetric flow rate (m³/s) you want in every zone.
SUPPLY_TEMP_C   - AHU supply air temperature override (°C).
"""

# ── 0. Environment setup ──────────────────────────────────────────────────────
!pip uninstall -y energy-plus-utility -q
!pip install -q "energy-plus-utility @ git+https://github.com/janithcyapa/energy-plus-utility.git@main"

from eplus import prepare_colab_eplus
prepare_colab_eplus(silent=False)

# ── 1. Imports ────────────────────────────────────────────────────────────────
import types, re, requests
from pathlib import Path

from eplus.core import EPlusUtil
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── 2. Control knob – change this one number ─────────────────────────────────
FIXED_FLOW_M3S  = 0.10   # m³/s per zone  (try 0.05 → 0.20)
SUPPLY_TEMP_C   = 14.0   # °C  – AHU discharge temperature

ZONES = ["SPACE1-1", "SPACE2-1", "SPACE3-1", "SPACE4-1", "SPACE5-1"]

# ── 3. Simulator setup ────────────────────────────────────────────────────────
OUT_DIR = "/simulation/eplus_out"
url_idf = "https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/5ZoneAirCooled.idf"
url_epw = "https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/Weather%20Files/LKA_Colombo-Katunayake.434500_SWERA.epw"

sim = EPlusUtil(verbose=0, out_dir=OUT_DIR)
sim.reset_state()
sim.delete_out_dir()
sim.clear_eplus_outputs(patterns="eplusout.*")
sim.set_model_from_url(url_idf, url_epw)

# ── 4. Patch IDF (Colombo DDY + 24/7 schedules) ───────────────────────────────
with open(sim.idf, 'r', encoding='utf-8') as f:
    idf_text = f.read()

idf_text = sim._remove_object_blocks(idf_text, "Site:Location")
idf_text = sim._remove_object_blocks(idf_text, "SizingPeriod:DesignDay")

for sched in ["FanAvailSched", "CoolingCoilAvailSched", "ReheatCoilAvailSched"]:
    pattern = re.compile(
        rf"(?i)^\s*Schedule:Compact[,\s]+{sched}[,\s].*?;[^\n]*\n?",
        re.MULTILINE | re.DOTALL
    )
    idf_text = pattern.sub("\n", idf_text)

url_ddy = "https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/Weather%20Files/LKA_Colombo-Katunayake.434500_SWERA.ddy"
ddy_text = requests.get(url_ddy).text

new_schedules = """
Schedule:Compact, FanAvailSched,          Fraction, Through: 12/31, For: AllDays, Until: 24:00, 1.0;
Schedule:Compact, CoolingCoilAvailSched,  Fraction, Through: 12/31, For: AllDays, Until: 24:00, 1.0;
Schedule:Compact, ReheatCoilAvailSched,   Fraction, Through: 12/31, For: AllDays, Until: 24:00, 1.0;
"""

idf_text = sim._append_block(idf_text, ddy_text)
idf_text = sim._append_block(idf_text, new_schedules)

with open(sim.idf, 'w', encoding='utf-8') as f:
    f.write(idf_text)

print("IDF patched ✓")

# ── 5. Dry run ────────────────────────────────────────────────────────────────
sim.ensure_output_sqlite()
sim.prepare_run_with_co2(outdoor_co2_ppm=420.0, wipe_outputs=True,
                          activate=True, reset=True)

# Request the variables we want to log
specs = [
    {"name": "Zone Mean Air Temperature",              "key": "*"},
    {"name": "Zone Air CO2 Concentration",             "key": "*"},
    {"name": "Zone People Occupant Count",             "key": "*"},
    {"name": "Site Outdoor Air Drybulb Temperature",   "key": "*"},
    # This is the node AFTER the VAV box → what actually enters the zone
    {"name": "System Node Current Density Volume Flow Rate", "key": "*"},
    {"name": "System Node Temperature",                "key": "*"},
]
sim.ensure_output_variables(specs, activate=True)

print("Running dry run …")
sim.run_dry_run(include_ems_edd=False, reset=True, design_day=True)
print("Dry run complete ✓")

# ── 6. Data logger ────────────────────────────────────────────────────────────
sim.collected_data = []

def state_logger(self, state):
    if not self.exchange.api_data_fully_ready(state):
        return

    day  = self.exchange.day_of_year(state)
    time = self.exchange.current_time(state)
    total_min = int(time * 60)
    hours, mins = divmod(total_min, 60)

    row = {
        "timestamp": f"Day {day:03d} {hours:02d}:{mins:02d}",
        "day": day, "hour": hours, "minute": mins,
    }

    t_out_h = self.exchange.get_variable_handle(
        state, "Site Outdoor Air Drybulb Temperature", "Environment")
    row["T_out"] = self.exchange.get_variable_value(state, t_out_h)

    for zone in ZONES:
        t_in_h  = self.exchange.get_variable_handle(state, "Zone Mean Air Temperature", zone)
        co2_h   = self.exchange.get_variable_handle(state, "Zone Air CO2 Concentration", zone)
        occ_h   = self.exchange.get_variable_handle(state, "Zone People Occupant Count", zone)
        # Actual flow delivered INTO the zone (after the VAV box + coil)
        flow_h  = self.exchange.get_variable_handle(
            state, "System Node Current Density Volume Flow Rate", f"{zone} In Node")

        row[f"{zone}_T_in"]  = self.exchange.get_variable_value(state, t_in_h)
        row[f"{zone}_CO2"]   = self.exchange.get_variable_value(state, co2_h)
        row[f"{zone}_Occ"]   = self.exchange.get_variable_value(state, occ_h)
        row[f"{zone}_Vdot"]  = self.exchange.get_variable_value(state, flow_h)

    self.collected_data.append(row)

sim.state_logger = types.MethodType(state_logger, sim)
sim.register_handlers("begin", [{"method_name": "state_logger"}])

# ── 7. Fixed-flow actuator handler ───────────────────────────────────────────
#
#  Strategy: clamp both the MIN and MAX available setpoints on the ATU inlet
#  node so the EnergyPlus VAV controller has no room to deviate.
#  We also push the thermostat setpoints far apart so the native PI loop
#  does not fight us.
#
def fixed_flow_controller(self, state):
    if not self.exchange.api_data_fully_ready(state):
        return

    # ── A. Disable the native heating/cooling PI loops ──────────────────────
    #  Push heating setpoint to -50 °C  → system never tries to heat
    #  Push cooling setpoint to +50 °C  → system never tries to cool
    self.tick_set_actuator(state,
        component_type="Schedule", control_type="Schedule Value",
        actuator_key="Htg-SetP-Sch", value=-50.0, allow_warmup=False)
    self.tick_set_actuator(state,
        component_type="Schedule", control_type="Schedule Value",
        actuator_key="Clg-SetP-Sch", value=50.0, allow_warmup=False)

    # ── B. Fix AHU discharge temperature ────────────────────────────────────
    self.tick_set_actuator(state,
        component_type="Schedule", control_type="Schedule Value",
        actuator_key="Seasonal Reset Supply Air Temp Sch",
        value=SUPPLY_TEMP_C, allow_warmup=False)

    # ── C. Convert volumetric → mass flow (ρ_air ≈ 1.204 kg/m³) ────────────
    rho_air  = 1.204
    m_dot    = FIXED_FLOW_M3S * rho_air    # kg/s

    # ── D. Clamp every VAV ATU inlet node ───────────────────────────────────
    #
    #  Node topology:
    #    AHU outlet → splitter → [SPACE1-1 ATU In Node] → VAV box
    #                                                    → reheat coil
    #                                                    → [SPACE1-1 In Node] → zone
    #
    #  We clamp the ATU In Node (upstream of the box), forcing the VAV damper
    #  to pass exactly m_dot regardless of the PI controller signal.
    #
    for zone in ZONES:
        node = f"{zone} ATU In Node"

        # Pin min = max = desired flow → no freedom for the damper
        self.tick_set_actuator(state,
            component_type="System Node Setpoint",
            control_type="Mass Flow Rate Minimum Available Setpoint",
            actuator_key=node, value=m_dot, allow_warmup=False)

        self.tick_set_actuator(state,
            component_type="System Node Setpoint",
            control_type="Mass Flow Rate Maximum Available Setpoint",
            actuator_key=node, value=m_dot, allow_warmup=False)

        self.tick_set_actuator(state,
            component_type="System Node Setpoint",
            control_type="Mass Flow Rate Setpoint",
            actuator_key=node, value=m_dot, allow_warmup=False)

sim.fixed_flow_controller = types.MethodType(fixed_flow_controller, sim)

# IMPORTANT: register on "inside_iter" so we run inside every HVAC iteration
sim.register_handlers("inside_iter", [{"method_name": "fixed_flow_controller"}])

print(f"Handlers registered ✓  (fixed flow = {FIXED_FLOW_M3S} m³/s = {FIXED_FLOW_M3S*1.204:.4f} kg/s per zone)")

# ── 8. Run simulation (Jan 1–7) ───────────────────────────────────────────────
sim.set_simulation_params(
    start=(1, 1),
    end=(1, 7),
    timestep_per_hour=12,           # 5-min steps
    start_day_of_week="Sunday",
)

print("\nStarting fixed-flow simulation …")
res = sim.run_annual()

if res == 0:
    print("Simulation complete ✓")
    df = pd.DataFrame(sim.collected_data)
    sim_start = pd.Timestamp("2026-01-01")
    df["datetime"] = (sim_start
                      + pd.to_timedelta(df["day"] - 1, unit="D")
                      + pd.to_timedelta(df["hour"] + df["minute"] / 60, unit="h"))
    df.set_index("datetime", inplace=True)
    print(df[[f"{ZONES[0]}_T_in", f"{ZONES[0]}_Vdot", f"{ZONES[0]}_CO2"]].head(20))
else:
    err_path = Path(OUT_DIR) / "eplusout.err"
    if err_path.exists():
        print("─── EnergyPlus error log (last 4000 chars) ───")
        print(open(err_path).read()[-4000:])

# ── 9. Quick diagnostic plot ──────────────────────────────────────────────────
if res == 0:
    fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                        subplot_titles=["Zone Temperature (°C)",
                                        "Delivered Airflow (m³/s)",
                                        "CO₂ Concentration (ppm)"])

    colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]

    for i, zone in enumerate(ZONES):
        c = colors[i]
        fig.add_trace(go.Scatter(x=df.index, y=df[f"{zone}_T_in"],
                                  name=zone, line=dict(color=c)), row=1, col=1)
        fig.add_trace(go.Scatter(x=df.index, y=df[f"{zone}_Vdot"],
                                  name=zone, line=dict(color=c), showlegend=False),
                      row=2, col=1)
        fig.add_trace(go.Scatter(x=df.index, y=df[f"{zone}_CO2"],
                                  name=zone, line=dict(color=c), showlegend=False),
                      row=3, col=1)

    # Reference line showing the target flow
    fig.add_hline(y=FIXED_FLOW_M3S, line_dash="dash", line_color="white",
                  annotation_text=f"Target = {FIXED_FLOW_M3S} m³/s", row=2, col=1)

    fig.add_hline(y=1000, line_dash="dot", line_color="red",
                  annotation_text="CO₂ limit 1000 ppm", row=3, col=1)

    fig.update_layout(height=900,
                      title_text=f"Fixed-Flow Control Validation (flow = {FIXED_FLOW_M3S} m³/s)",
                      template="plotly_dark", hovermode="x unified")
    fig.update_yaxes(title_text="°C",    row=1, col=1)
    fig.update_yaxes(title_text="m³/s",  row=2, col=1)
    fig.update_yaxes(title_text="ppm",   row=3, col=1)
    fig.show()

    # ── Summary ────────────────────────────────────────────────────────────
    print("\n── Flow delivery summary ─────────────────────────────────────────")
    for zone in ZONES:
        col = df[f"{zone}_Vdot"]
        print(f"  {zone}: mean={col.mean():.4f}  min={col.min():.4f}  "
              f"max={col.max():.4f}  target={FIXED_FLOW_M3S:.4f}  m³/s")
    print("─" * 66)
    print("\nIf mean ≈ target the actuator is working correctly.")
    print("If mean ≈ 0, the ATU node name may not match — run sim.list_available_actuators() to check.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
IDF patched ✓
Running dry run …
Dry run complete ✓
Handlers registered ✓  (fixed flow = 0.1 m³/s = 0.1204 kg/s per zone)

Starting fixed-flow simulation …
Simulation complete ✓
                               SPACE1-1_T_in  SPACE1-1_Vdot  SPACE1-1_CO2
datetime                                                                 
2026-01-01 00:04:59.999999998      22.594918            0.0         420.0
2026-01-01 00:10:00.000000001      22.574281            0.0         420.0
2026-01-01 00:15:00.000000000      22.551213            0.0         420.0
2026-01-01 00:19:59.999999998      22.526374            0.0         420.0
2026-01-01 00:24:00.000000000      22.500549            0.0         420.0
2026-01-01 00:30:00.000000000      22.474238            0.0         420.0
2026-01-01 00:34:00.000000001      22.447718            0.0         420.0
2026-01-01 00:40:00.00


── Flow delivery summary ─────────────────────────────────────────
  SPACE1-1: mean=0.0338  min=0.0000  max=0.1010  target=0.1000  m³/s
  SPACE2-1: mean=0.0338  min=0.0000  max=0.1014  target=0.1000  m³/s
  SPACE3-1: mean=0.0338  min=0.0000  max=0.1010  target=0.1000  m³/s
  SPACE4-1: mean=0.0338  min=0.0000  max=0.1016  target=0.1000  m³/s
  SPACE5-1: mean=0.0338  min=0.0000  max=0.1010  target=0.1000  m³/s
──────────────────────────────────────────────────────────────────

If mean ≈ target the actuator is working correctly.
If mean ≈ 0, the ATU node name may not match — run sim.list_available_actuators() to check.
